# 08 - Baseline vs. Pipeline — End-to-End Comparison (ContraDoc)

Runs the **text-dump baseline**: for each document, send the full numbered sentence list to the LLM in a single call and ask it to identify every contradictory sentence pair. Then compares against the **full KG pipeline** (steps 02 → 07) on the same 100-doc corpus.

## Why this baseline

It answers the most direct question a reader will ask: *"Why bother with the KG, retrieval, and NLI stages if I can just throw the document at an LLM?"* Both sides get the same text and the same LLM (`settings.llm_model`), so any difference is attributable to the pipeline itself.

**Baseline inputs:** numbered sentences from `triples.jsonl` (same sentence split the pipeline uses — ensures sentence_ids align).

**Outputs:**
- `data/processed/ContraDoc/baseline_predictions.jsonl` — one record per predicted contradictory pair (raw-text → LLM)
- Comparison table at the end of the notebook.

In [1]:
import json
from collections import Counter
from pathlib import Path
from typing import Literal

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from config import settings

TRIPLES_PATH = Path("data/processed/ContraDoc/triples.jsonl")
PIPELINE_OUTPUT = Path("data/processed/ContraDoc/llm_judgments.jsonl")
BASELINE_OUTPUT = Path("data/processed/ContraDoc/baseline_predictions.jsonl")

## Load per-doc sentences + gold pairs

In [2]:
records = [json.loads(line) for line in TRIPLES_PATH.open(encoding="utf-8")]
print(f"Loaded {len(records)} docs from {TRIPLES_PATH}")

docs = {}
for rec in records:
    doc_id = rec["doc_id"]
    docs[doc_id] = {
        "contradiction": rec["contradiction"],
        "gold_evidence_sentence_id": rec.get("gold_evidence_sentence_id"),
        "gold_ref_sentence_ids": rec.get("gold_ref_sentence_ids") or [],
        "sentences": [(s["sentence_id"], s["source_text"]) for s in rec["sentences"]],
    }


def pkey(a, b):
    return (a, b) if a < b else (b, a)


# Build gold chunk pair set in the same format both sides compare against
gold_pairs = set()
for doc_id, d in docs.items():
    ev = d["gold_evidence_sentence_id"]
    refs = d["gold_ref_sentence_ids"]
    if d["contradiction"] != "YES" or ev is None or not refs:
        continue
    for ref in refs:
        if ev == ref:
            continue  # skip intra (ev == ref)
        gold_pairs.add(pkey((doc_id, ev), (doc_id, ref)))

n_gold_docs = len({p[0][0] for p in gold_pairs})
print(f"Gold chunk pairs: {len(gold_pairs)} across {n_gold_docs} docs (intra-cases excluded)")

Loaded 100 docs from data\processed\ContraDoc\triples.jsonl
Gold chunk pairs: 31 across 31 docs (intra-cases excluded)


## Structured-output schema for the baseline

The LLM returns a list of contradictory pairs it found in the document. Each pair references two sentence IDs (matching the numbering we put in the prompt).

In [3]:
class BaselinePair(BaseModel):
    sentence_id_a: int = Field(description="Sentence ID of the first sentence in the contradicting pair (1-indexed, as given in the prompt).")
    sentence_id_b: int = Field(description="Sentence ID of the second sentence in the contradicting pair.")
    contradiction_type: Literal[
        "Negation",
        "Numeric",
        "Content",
        "Perspective/View/Opinion",
        "Emotion/Mood/Feeling",
        "Relation",
        "Factual",
        "Causal",
    ] = Field(description="Most applicable contradiction type.")
    confidence: Literal["low", "medium", "high"] = Field(description="Self-assessed confidence.")
    explanation: str = Field(description="One-sentence rationale naming the specific conflict.")


class BaselineResponse(BaseModel):
    contradictions: list[BaselinePair] = Field(
        description="Every contradictory sentence pair found in the document. Empty list if no contradictions."
    )

## LLM client and prompt

Same ContraDoc taxonomy as step 07, presented as a multi-pair task over the whole document.

In [4]:
SYSTEM_PROMPT = """You are given a document with numbered sentences. Identify every pair of sentences in the document that contradict each other.

A PAIR IS CONTRADICTORY when:
- Both sentences describe the same entity/event/relationship but assert incompatible attributes.
- They cannot both be true in the same document context (ignoring obvious narrative time-series where attributes legitimately change).

A PAIR IS NOT CONTRADICTORY when:
- The sentences talk about different entities, different works, or different time points that can coexist.
- One sentence is a specialization or paraphrase of the other.
- They share vocabulary but assert compatible facts.

Classify each contradiction into ONE type (ContraDoc taxonomy):
- Negation: explicit negation flips polarity.
- Numeric: number, date, or quantity mismatch about the same entity.
- Content: non-numeric, non-subjective attribute swap.
- Perspective/View/Opinion: conflicting subjective evaluations.
- Emotion/Mood/Feeling: conflicting emotional states.
- Relation: mutually exclusive relations between entities.
- Factual: real-world factual mismatch.
- Causal: effect does not match stated cause.

Return the full list of contradictory pairs (sentence_id_a, sentence_id_b, type, confidence, explanation). If the document contains no contradictions, return an empty list. Do NOT invent pairs that are not genuinely contradictory.
"""

llm = ChatOpenAI(model=settings.llm_model, api_key=settings.openai_api_key.get_secret_value(), temperature=0)
baseline_judge = llm.with_structured_output(BaselineResponse)


def format_doc(sentences):
    return "\n".join(f"[{sid}] {text}" for sid, text in sentences)


def run_baseline(sentences) -> BaselineResponse:
    user_msg = f"Document:\n{format_doc(sentences)}\n\nReturn the full list of contradictions."
    return baseline_judge.invoke(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ]
    )

## Sanity check on one gold-pair YES doc

In [5]:
sample_doc_id = next(iter({p[0][0] for p in gold_pairs}))
sample = docs[sample_doc_id]
print(f"Testing on doc_id={sample_doc_id} (gold ev_sid={sample['gold_evidence_sentence_id']}, ref_sids={sample['gold_ref_sentence_ids']})")

resp = run_baseline(sample["sentences"])
print(f"\nBaseline returned {len(resp.contradictions)} contradiction(s):")
for p in resp.contradictions:
    print(f"  ({p.sentence_id_a}, {p.sentence_id_b}) type={p.contradiction_type} conf={p.confidence}")
    print(f"    {p.explanation}")

Testing on doc_id=3488771872_3 (gold ev_sid=5, ref_sids=[4])

Baseline returned 2 contradiction(s):
  (3, 4) type=Relation conf=high
    Sentence 3 says Brown is expected to propose to Phoebe, while sentence 4 says that when he arrives he does not propose and instead announces he is leaving to fight in Europe.
  (13, 15) type=Content conf=high
    Sentence 13 says Brown seems to prefer Miss Livvy over Phoebe's true personality, but sentence 15 says he has discovered his love is for Miss Phoebe and not Miss Livvy.


## Run baseline on every document (resumable)

One LLM call per doc → expect ~100 calls total, ~3-5 minutes.

In [6]:
BASELINE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

done_doc_ids: set[str] = set()
if BASELINE_OUTPUT.exists():
    with BASELINE_OUTPUT.open(encoding="utf-8") as f:
        for line in f:
            done_doc_ids.add(json.loads(line)["doc_id"])
    print(f"Resuming: {len(done_doc_ids)} docs already baselined")

todo = [d for d in docs if d not in done_doc_ids]
print(f"To baseline: {len(todo)} docs")

with BASELINE_OUTPUT.open("a", encoding="utf-8") as f:
    for doc_id in tqdm(todo):
        d = docs[doc_id]
        try:
            resp = run_baseline(d["sentences"])
        except Exception as exc:
            print(f"FAILED {doc_id}: {type(exc).__name__}: {exc}")
            continue
        text_by_sid = dict(d["sentences"])
        for p in resp.contradictions:
            rec = {
                "doc_id": doc_id,
                "contradiction_label": d["contradiction"],
                "sentence_id_a": p.sentence_id_a,
                "sentence_id_b": p.sentence_id_b,
                "source_text_a": text_by_sid.get(p.sentence_id_a, ""),
                "source_text_b": text_by_sid.get(p.sentence_id_b, ""),
                "contradiction_type": p.contradiction_type,
                "confidence": p.confidence,
                "explanation": p.explanation,
                "is_gold_pair": pkey((doc_id, p.sentence_id_a), (doc_id, p.sentence_id_b)) in gold_pairs,
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        if not resp.contradictions:
            # Write a 'no-contradictions-found' marker so the resume logic counts this doc as done
            f.write(
                json.dumps({"doc_id": doc_id, "contradiction_label": d["contradiction"], "sentence_id_a": None, "sentence_id_b": None}) + "\n"
            )
        f.flush()

print(f"Done. Output: {BASELINE_OUTPUT.resolve()}")

To baseline: 100 docs


  0%|          | 0/100 [00:00<?, ?it/s]

Done. Output: D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\ContraDoc\baseline_predictions.jsonl


## End-to-end comparison: Baseline vs Pipeline

Both predict sets of chunk pairs (represented by `(doc_id, sid)` tuples). Evaluate against the same gold chunk pair set.

In [7]:
# --- Baseline predictions: chunk pairs ---
baseline_records = [json.loads(line) for line in BASELINE_OUTPUT.open(encoding="utf-8")]
baseline_pairs = set()
for r in baseline_records:
    if r["sentence_id_a"] is None:
        continue
    baseline_pairs.add(pkey((r["doc_id"], r["sentence_id_a"]), (r["doc_id"], r["sentence_id_b"])))

# --- Pipeline predictions: chunk pairs where llm_is_contradiction == True ---
pipeline_pairs = set()
if PIPELINE_OUTPUT.exists():
    pipeline_records = [json.loads(line) for line in PIPELINE_OUTPUT.open(encoding="utf-8")]
    for r in pipeline_records:
        if r.get("llm_is_contradiction"):
            pipeline_pairs.add(pkey((r["doc_id"], r["chunk_a"]["sentence_id"]), (r["doc_id"], r["chunk_b"]["sentence_id"])))
    n_pipeline_calls = len(pipeline_records)
else:
    print("WARN: pipeline output (step 07) not found — skipping pipeline side of comparison.")
    pipeline_records = []
    n_pipeline_calls = 0


def metrics(pairs, name, n_llm_calls):
    tp = len(pairs & gold_pairs)
    fp = len(pairs - gold_pairs)
    fn = len(gold_pairs - pairs)
    prec = tp / max(tp + fp, 1)
    rec = tp / max(len(gold_pairs), 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)
    docs_caught = {p[0][0] for p in pairs & gold_pairs}
    doc_r = len(docs_caught) / max(n_gold_docs, 1)
    return {
        "name": name,
        "n_pred": len(pairs),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "prec": prec,
        "rec": rec,
        "f1": f1,
        "doc_r": doc_r,
        "docs_caught": len(docs_caught),
        "n_llm_calls": n_llm_calls,
    }


n_baseline_calls = len(docs)  # one call per doc
rows = [metrics(baseline_pairs, "Baseline (text dump -> LLM)", n_baseline_calls)]
if pipeline_pairs or pipeline_records:
    rows.append(metrics(pipeline_pairs, "Pipeline (KG + NLI + LLM)", n_pipeline_calls))

print(f"Gold chunk pairs: {len(gold_pairs)} across {n_gold_docs} docs")
print()
header = f"{'Method':32} {'#pred':>6}  {'TP':>3}  {'FP':>4}  {'FN':>3}  {'Prec':>6}  {'Pair-R':>6}  {'F1':>6}  {'Doc-R':>11}  {'LLM calls':>10}"
print(header)
print("-" * len(header))
for r in rows:
    print(
        f"{r['name']:32} {r['n_pred']:>6}  {r['tp']:>3}  {r['fp']:>4}  {r['fn']:>3}  "
        f"{r['prec']:>5.1%}  {r['rec']:>5.1%}  {r['f1']:>5.1%}  "
        f"{r['docs_caught']:>2}/{n_gold_docs:<2} {r['doc_r']:>5.1%}  "
        f"{r['n_llm_calls']:>10}"
    )

# --- Overlap: caught by both / baseline only / pipeline only ---
if pipeline_pairs:
    both = (baseline_pairs & gold_pairs) & (pipeline_pairs & gold_pairs)
    baseline_only = (baseline_pairs & gold_pairs) - pipeline_pairs
    pipeline_only = (pipeline_pairs & gold_pairs) - baseline_pairs
    print()
    print("=== Gold pairs caught — overlap ===")
    print(f"  Both: {len(both)}")
    print(f"  Baseline only: {len(baseline_only)}")
    print(f"  Pipeline only: {len(pipeline_only)}")

Gold chunk pairs: 31 across 31 docs

Method                            #pred   TP    FP   FN    Prec  Pair-R      F1        Doc-R   LLM calls
--------------------------------------------------------------------------------------------------------
Baseline (text dump -> LLM)         113   15    98   16  13.3%  48.4%  20.8%  15/31 48.4%         100
Pipeline (KG + NLI + LLM)           149   18   131   13  12.1%  58.1%  20.0%  18/31 58.1%         581

=== Gold pairs caught — overlap ===
  Both: 12
  Baseline only: 3
  Pipeline only: 6


## Per-type breakdown and disagreement samples

In [8]:
baseline_type_dist = Counter(r["contradiction_type"] for r in baseline_records if r.get("sentence_id_a") is not None)
print("=== Baseline predicted-type distribution ===")
for t, n in baseline_type_dist.most_common():
    print(f"  {t}: {n}")

print("\n=== Sample baseline-only caught gold pairs (pipeline missed) ===")
if pipeline_pairs:
    for p in list(baseline_only)[:3]:
        (d, sa), (_, sb) = p
        a_text = dict(docs[d]["sentences"])[sa]
        b_text = dict(docs[d]["sentences"])[sb]
        print(f"  doc={d}")
        print(f"    A [sid={sa}]: {a_text[:140]}")
        print(f"    B [sid={sb}]: {b_text[:140]}")

print("\n=== Sample pipeline-only caught gold pairs (baseline missed) ===")
if pipeline_pairs:
    for p in list(pipeline_only)[:3]:
        (d, sa), (_, sb) = p
        a_text = dict(docs[d]["sentences"])[sa]
        b_text = dict(docs[d]["sentences"])[sb]
        print(f"  doc={d}")
        print(f"    A [sid={sa}]: {a_text[:140]}")
        print(f"    B [sid={sb}]: {b_text[:140]}")

=== Baseline predicted-type distribution ===
  Content: 55
  Numeric: 30
  Factual: 14
  Negation: 8
  Relation: 5
  Perspective/View/Opinion: 1

=== Sample baseline-only caught gold pairs (pipeline missed) ===
  doc=3488771839_11
    A [sid=3]: Mrs. Hudson says that Holmes has neither eaten nor drunk anything in three days.
    B [sid=29]: Eating normally for three days, and the claim of the "disease's" infectious nature was to keep Watson from examining him and discovering the
  doc=3503017474_2
    A [sid=3]: The rhyme dates back at least to the 16th century.
    B [sid=5]: The rhyme dates back at least to the 18th century and exists with different numbers of verses each with a number of variations.
  doc=3489738225_1
    A [sid=4]: Charleston is suspected of involvement in 15 commercial robberies dating to November 2013, according to FBI officials.
    B [sid=18]: Although he declined to give details of the 32 previous robberies, Emmett said it was an intensive investigation that w

## Per-type recall — Baseline vs Pipeline side-by-side

Which methodology catches which contradiction types? Multi-label docs contribute to every listed type.

In [9]:
from collections import defaultdict

# Doc types already available from `records` / `docs` loaded earlier (contra_type field)
doc_types = {}
for rec in records:
    if rec["contradiction"] == "YES":
        doc_types[rec["doc_id"]] = [t for t in (rec.get("contra_type") or "none").split("|") if t]


def per_type_recall_side_by_side():
    all_types = sorted({t for ts in doc_types.values() for t in ts}, key=lambda x: -sum(1 for ts in doc_types.values() if x in ts))

    # Compute per-type stats for each method
    def counts_for(pairs):
        totals = defaultdict(int)
        caught = defaultdict(int)
        for p in gold_pairs:
            doc_id = p[0][0]
            for t in doc_types.get(doc_id, ["unknown"]):
                totals[t] += 1
                if p in pairs:
                    caught[t] += 1
        return totals, caught

    b_tot, b_cat = counts_for(baseline_pairs)
    p_tot, p_cat = counts_for(pipeline_pairs)

    print(f"  {'type':30s} | {'Baseline':>15} | {'Pipeline':>15}")
    print(f"  {'':30s} | {'caught/total  R':>15} | {'caught/total  R':>15}")
    print("  " + "-" * 72)
    for t in all_types:
        bt, bc = b_tot[t], b_cat[t]
        pt, pc = p_tot[t], p_cat[t]
        b_cell = f"{bc:>3}/{bt:<3} {bc / max(bt, 1):>5.1%}" if bt else "     -       "
        p_cell = f"{pc:>3}/{pt:<3} {pc / max(pt, 1):>5.1%}" if pt else "     -       "
        print(f"  {t:30s} | {b_cell:>15} | {p_cell:>15}")


if pipeline_pairs:
    per_type_recall_side_by_side()
else:
    # Baseline-only breakdown if pipeline wasn't run yet
    def per_type_recall(pairs, name):
        type_totals = defaultdict(int)
        type_caught = defaultdict(int)
        for p in gold_pairs:
            doc_id = p[0][0]
            for t in doc_types.get(doc_id, ["unknown"]):
                type_totals[t] += 1
                if p in pairs:
                    type_caught[t] += 1
        all_types = sorted(type_totals.keys(), key=lambda x: -type_totals[x])
        print(f"\n{name}:")
        print(f"  {'type':30s}  caught  total  recall")
        print("  " + "-" * 52)
        for t in all_types:
            caught, total = type_caught[t], type_totals[t]
            print(f"  {t:30s}  {caught:>6}  {total:>5}  {caught / max(total, 1):>6.1%}")

    per_type_recall(baseline_pairs, "Baseline (text-dump -> LLM)")

  type                           |        Baseline |        Pipeline
                                 | caught/total  R | caught/total  R
  ------------------------------------------------------------------------
  Content                        |    12/20  60.0% |    13/20  65.0%
  Numeric                        |     5/9   55.6% |     6/9   66.7%
  Perspective/View/Opinion       |     2/5   40.0% |     4/5   80.0%
  Negation                       |     3/5   60.0% |     3/5   60.0%
  Factual                        |    2/2   100.0% |    2/2   100.0%
  Emotion/Mood/Feeling           |     0/4    0.0% |     0/4    0.0%
  Relation                       |     0/1    0.0% |     0/1    0.0%
  Causal                         |     0/1    0.0% |     0/1    0.0%
